# Setup

In [13]:
checkpoint = "prajjwal1/bert-tiny"
tokenizer_checkpoint = "bert-base-uncased"
dataset_name = "imdb"


from chop.tools import get_tokenized_dataset

dataset, tokenizer = get_tokenized_dataset(
    dataset=dataset_name,
    checkpoint=tokenizer_checkpoint,
    return_tokenizer=True,
)


# Defining search space

import torch.nn as nn
from chop.nn.modules import Identity

search_space = {
    "num_layers": [2, 4, 8],
    "num_heads": [2, 4, 8, 16],
    "hidden_size": [128, 192, 256, 384, 512],
    "intermediate_size": [512, 768, 1024, 1536, 2048],
    "linear_layer_choices": [
        nn.Linear,
        Identity,
    ],
}

# model constructor

from transformers import AutoConfig, AutoModelForSequenceClassification
from chop.tools.utils import deepsetattr


def construct_model(trial):
    config = AutoConfig.from_pretrained(checkpoint)

    # Update the paramaters in the config
    for param in [
        "num_layers",
        "num_heads",
        "hidden_size",
        "intermediate_size",
    ]:
        chosen_idx = trial.suggest_int(param, 0, len(search_space[param]) - 1)
        setattr(config, param, search_space[param][chosen_idx])

    trial_model = AutoModelForSequenceClassification.from_config(config)

    for name, layer in trial_model.named_modules():
        if isinstance(layer, nn.Linear) and layer.in_features == layer.out_features:
            new_layer_cls = trial.suggest_categorical(
                f"{name}_type",
                search_space["linear_layer_choices"],
            )

            if new_layer_cls == nn.Linear:
                continue
            elif new_layer_cls == Identity:
                new_layer = Identity()
                deepsetattr(trial_model, name, new_layer)
            else:
                raise ValueError(f"Unknown layer type: {new_layer_cls}")

    return trial_model




INFO     Tokenizing dataset imdb with AutoTokenizer for bert-base-uncased.


# Objective Function

In [14]:

# Trainer
from chop.tools import get_trainer


def objective(trial):

    # Define the model
    model = construct_model(trial)

    trainer = get_trainer(
        model=model,
        tokenized_dataset=dataset,
        tokenizer=tokenizer,
        evaluate_metric="accuracy",
        num_train_epochs=1,
    )

    trainer.train()
    eval_results = trainer.evaluate()

    # Set the model as an attribute so we can fetch it later
    trial.set_user_attr("model", model)

    return eval_results["eval_accuracy"]



# Sampler

In [15]:
from optuna.samplers import GridSampler, RandomSampler, TPESampler

sampler = RandomSampler()

import optuna
import joblib



# Random Sampler
def save_checkpoint_random(study, trial):
    joblib.dump(study, "tutorial-5-study-random.pkl")
    
    model = trial.user_attrs["model"]
    num_params = sum(p.numel() for p in model.parameters())
    
    print(f"Trial {trial.number} finished with value: {trial.value}")
    print(f"  Params: {trial.params}")
    print(f"  Model parameters: {num_params:,}")
    print(f"  Best so far: {study.best_value} (Trial {study.best_trial.number})")

study_random = optuna.create_study(
    direction="maximize",
    study_name="tutorial-5-study-random",
    sampler=sampler,
)

study_random.optimize(
    objective,
    n_trials=2,
    timeout=60 * 60 * 24,
    callbacks=[save_checkpoint_random],
)


[I 2026-02-02 16:38:32,878] A new study created in memory with name: tutorial-5-study-random
/home/ism/ADL/mase/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.696600
1000,0.656300
1500,0.528300
2000,0.461200
2500,0.424100
3000,0.414400


[I 2026-02-02 16:39:30,037] Trial 0 finished with value: 0.83136 and parameters: {'num_layers': 1, 'num_heads': 3, 'hidden_size': 2, 'intermediate_size': 3, 'bert.encoder.layer.0.attention.self.query_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.0.attention.self.key_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.0.attention.self.value_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.0.attention.output.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.self.query_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.1.attention.self.key_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.1.attention.self.value_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.1.attention.output.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.pooler.dense_type': <class 'torch.nn.modules.linear.Linear'>}. Best is trial 0 with

Trial 0 finished with value: 0.83136
  Params: {'num_layers': 1, 'num_heads': 3, 'hidden_size': 2, 'intermediate_size': 3, 'bert.encoder.layer.0.attention.self.query_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.0.attention.self.key_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.0.attention.self.value_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.0.attention.output.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.self.query_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.1.attention.self.key_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.1.attention.self.value_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.1.attention.output.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.pooler.dense_type': <class 'torch.nn.modules.linear.Linear'>}
  Model parameters: 9,722,114
  Best so far: 0.83136 (T

/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'torch.nn.modules.linear.Linear'> which is of type type.
  warnings.warn(message)
/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'chop.nn.modules.identity.Identity'> which is of type type.
  warnings.warn(message)
/home/ism/ADL/mase/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.693200
1000,0.601600
1500,0.454800
2000,0.401000
2500,0.360700
3000,0.367900


[I 2026-02-02 16:40:27,630] Trial 1 finished with value: 0.84636 and parameters: {'num_layers': 0, 'num_heads': 2, 'hidden_size': 0, 'intermediate_size': 1, 'bert.encoder.layer.0.attention.self.query_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.0.attention.self.key_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.0.attention.self.value_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.0.attention.output.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.self.query_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.self.key_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.self.value_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.output.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.pooler.dense_type': <class 'torch.nn.modules.linear.Linear'>}. Best is trial 1 with value: 0.84636.


Trial 1 finished with value: 0.84636
  Params: {'num_layers': 0, 'num_heads': 2, 'hidden_size': 0, 'intermediate_size': 1, 'bert.encoder.layer.0.attention.self.query_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.0.attention.self.key_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.0.attention.self.value_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.0.attention.output.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.self.query_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.self.key_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.self.value_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.output.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.pooler.dense_type': <class 'torch.nn.modules.linear.Linear'>}
  Model parameters: 4,517,762
  Best so far: 0.84636 (Trial 1)


In [ ]:

# GridSampler
def save_checkpoint_grid(study, trial):
    joblib.dump(study, "tutorial-5-study-grid.pkl")
    
    model = trial.user_attrs["model"]
    num_params = sum(p.numel() for p in model.parameters())
    
    print(f"Trial {trial.number} finished with value: {trial.value}")
    print(f"  Params: {trial.params}")
    print(f"  Model parameters: {num_params:,}")
    print(f"  Best so far: {study.best_value} (Trial {study.best_trial.number})")

sampler = GridSampler(search_space)
study_grid = optuna.create_study(
    direction="maximize",
    study_name="tutorial-5-study-grid",
    sampler=sampler,
)

study_grid.optimize(
    objective,
    n_trials=2,
    timeout=60 * 60 * 24,
    callbacks=[save_checkpoint_grid]
)


[I 2026-02-02 16:58:16,882] A new study created in memory with name: tutorial-5-study-grid
/home/ism/ADL/mase/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.665500
1000,0.499100
1500,0.403700
2000,0.359100
2500,0.319300
3000,0.350900


[I 2026-02-02 16:59:25,373] Trial 0 finished with value: 0.86772 and parameters: {'num_layers': 4, 'num_heads': 4, 'hidden_size': 384, 'intermediate_size': 1024}. Best is trial 0 with value: 0.86772.


Trial 0 finished with value: 0.86772
  Params: {'num_layers': 4, 'num_heads': 4, 'hidden_size': 384, 'intermediate_size': 1024}
  Model parameters: 14,828,674
  Best so far: 0.86772 (Trial 0)


/home/ism/ADL/mase/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.658100
1000,0.494100
1500,0.415500
2000,0.357300
2500,0.324000
3000,0.339700


[I 2026-02-02 17:00:32,655] Trial 1 finished with value: 0.86796 and parameters: {'num_layers': 8, 'num_heads': 8, 'hidden_size': 384, 'intermediate_size': 512}. Best is trial 1 with value: 0.86796.


Trial 1 finished with value: 0.86796
  Params: {'num_layers': 8, 'num_heads': 8, 'hidden_size': 384, 'intermediate_size': 512}
  Model parameters: 14,041,218
  Best so far: 0.86796 (Trial 1)


In [17]:


# TPESampler
def save_checkpoint_tpe(study, trial):
    joblib.dump(study, "tutorial-5-study-tpe.pkl")
    
    model = trial.user_attrs["model"]
    num_params = sum(p.numel() for p in model.parameters())
    
    print(f"Trial {trial.number} finished with value: {trial.value}")
    print(f"  Params: {trial.params}")
    print(f"  Model parameters: {num_params:,}")
    print(f"  Best so far: {study.best_value} (Trial {study.best_trial.number})")

sampler = TPESampler()
study_tpe = optuna.create_study(
    direction="maximize",
    study_name="tutorial-5-study-tpe",
    sampler=sampler,
)

study_tpe.optimize(
    objective,
    n_trials=2,
    timeout=60 * 60 * 24,
    callbacks=[save_checkpoint_tpe]
)

[I 2026-02-02 16:44:53,441] A new study created in memory with name: tutorial-5-study-tpe
/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'torch.nn.modules.linear.Linear'> which is of type type.
  warnings.warn(message)
/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'chop.nn.modules.identity.Identity'> which is of type type.
  warnings.warn(message)
/home/ism/ADL/mase/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.661800
1000,0.523800
1500,0.466400
2000,0.427400
2500,0.408900
3000,0.392700


[I 2026-02-02 16:46:08,382] Trial 0 finished with value: 0.8354 and parameters: {'num_layers': 0, 'num_heads': 1, 'hidden_size': 4, 'intermediate_size': 3, 'bert.encoder.layer.0.attention.self.query_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.0.attention.self.key_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.0.attention.self.value_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.0.attention.output.dense_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.1.attention.self.query_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.1.attention.self.key_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.1.attention.self.value_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.output.dense_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.pooler.dense_type': <class 'chop.nn.modules.identity.Identity'>}. Best is trial 0 wi

Trial 0 finished with value: 0.8354
  Params: {'num_layers': 0, 'num_heads': 1, 'hidden_size': 4, 'intermediate_size': 3, 'bert.encoder.layer.0.attention.self.query_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.0.attention.self.key_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.0.attention.self.value_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.0.attention.output.dense_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.1.attention.self.query_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.1.attention.self.key_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.1.attention.self.value_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.output.dense_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.pooler.dense_type': <class 'chop.nn.modules.identity.Identity'>}
  Model parameters: 19,571,714
  Best so far: 0.8354 

/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'torch.nn.modules.linear.Linear'> which is of type type.
  warnings.warn(message)
/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'chop.nn.modules.identity.Identity'> which is of type type.
  warnings.warn(message)
/home/ism/ADL/mase/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.677200
1000,0.507100
1500,0.405400
2000,0.359800
2500,0.326700
3000,0.339100


[I 2026-02-02 16:47:07,094] Trial 1 finished with value: 0.86228 and parameters: {'num_layers': 0, 'num_heads': 3, 'hidden_size': 2, 'intermediate_size': 2, 'bert.encoder.layer.0.attention.self.query_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.0.attention.self.key_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.0.attention.self.value_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.0.attention.output.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.self.query_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.self.key_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.self.value_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.output.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.pooler.dense_type': <class 'chop.nn.modules.identity.Identity'>}. Best is trial 1 with value: 0.86

Trial 1 finished with value: 0.86228
  Params: {'num_layers': 0, 'num_heads': 3, 'hidden_size': 2, 'intermediate_size': 2, 'bert.encoder.layer.0.attention.self.query_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.0.attention.self.key_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.0.attention.self.value_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.0.attention.output.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.self.query_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.self.key_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.self.value_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.output.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.pooler.dense_type': <class 'chop.nn.modules.identity.Identity'>}
  Model parameters: 9,459,970
  Best so far: 0.86228 (Trial 1)


# Save

In [22]:

# Not sure how to save it at the end
import joblib
joblib.dump(study_random, "./tutorial_5_task_1/study_random.pkl")
joblib.dump(study_grid, "./tutorial_5_task_1/study_grid.pkl")
joblib.dump(study_tpe, "./tutorial_5_task_1/study_tpe.pkl")


FileNotFoundError: [Errno 2] No such file or directory: './tutorial_5_task_1/study_random.pkl'